In [14]:
# import ydf
import pandas as pd
from ydata_profiling import ProfileReport
import sklearn as sk
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import seaborn as sns 

train=pd.read_csv("https://raw.githubusercontent.com/yani-iben/House-Prices-Prediction-using-TensorFlow-Decision-Forests/refs/heads/main/ames_data/hw5_train.csv")
test=pd.read_csv("https://raw.githubusercontent.com/yani-iben/House-Prices-Prediction-using-TensorFlow-Decision-Forests/refs/heads/main/ames_data/hw5_test.csv")

*Predicting House Prices*



In [15]:
# train_profile= ProfileReport(train,title="Train Data Profiling Report", html={'style':{'full_width':True}},minimal=True)

# test_profile= ProfileReport(test,title="Test Data Profiling Report", html={'style':{'full_width':True}},minimal=True)

# train_profile.to_file("train_profile.html")
# test_profile.to_file("test_profile.html")

In [16]:
# train_profile.to_notebook_iframe()

Dropping Variables with >85% Missing Values

In [17]:
cols_to_drop=["BsmtQual","MiscFeature","PoolQC","BsmtHalfBath","3SsnPorch","PoolArea","MiscVal","BsmtFinSF2","EnclosedPorch"]

train=train.drop(cols_to_drop,axis=1)
test=test.drop(cols_to_drop,axis=1)


***Feature Engineering***

In [18]:
train["Total_SF"]=train["TotalBsmtSF"]+train["1stFlrSF"]+train["2ndFlrSF"]
test["Total_SF"]=test["TotalBsmtSF"]+test["1stFlrSF"]+test["2ndFlrSF"]

train["Total_Bathrooms"]=train["FullBath"]+(0.5*train["HalfBath"])+train["BsmtFullBath"]
test["Total_Bathrooms"]=test["FullBath"]+(0.5*test["HalfBath"])+test["BsmtFullBath"]

train["Outdoor_Space"]=train["OpenPorchSF"]+train["WoodDeckSF"]+train["ScreenPorch"]
test["Outdoor_Space"]=test["OpenPorchSF"]+test["WoodDeckSF"]+test["ScreenPorch"]

train["House_Age"]=train["YrSold"]-train["YearBuilt"]
test["House_Age"]=test["YrSold"]-test["YearBuilt"]

train["Years_Since_Remodel"]=train["YrSold"]-train["YearRemodAdd"]
test["Years_Since_Remodel"]=test["YrSold"]-test["YearRemodAdd"]


In [19]:
train["Neighborhood"].unique()

array(['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst',
       'NWAmes', 'OldTown', 'BrkSide', 'Sawyer', 'NridgHt', 'NAmes',
       'SawyerW', 'IDOTRR', 'MeadowV', 'Edwards', 'Timber', 'Gilbert',
       'StoneBr', 'ClearCr', 'NPkVill', 'Blmngtn', 'BrDale', 'SWISU',
       'Blueste'], dtype=object)

In [20]:
nbhd_median_price=train.groupby("Neighborhood")["SalePrice"].median()


global_median=train["SalePrice"].median()

def apply_target_encoding(df,stats,fallback):
    return df["Neighborhood"].map(stats).fillna(fallback)




train["Nbhd_Price_Point"]=apply_target_encoding(train,nbhd_median_price,global_median)
test["Nbhd_Price_Point"]=apply_target_encoding(test,nbhd_median_price,global_median)




Still need to encode

In [21]:
# selecting all the categorical variables 
cat_variables = [var for var in train.columns if train[var].dtype == 'object']
print(cat_variables)

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'Fence', 'SaleType', 'SaleCondition']


In [22]:
# going to encode the categorical variables for the train set 
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
train[cat_variables] = encoder.fit_transform(train[cat_variables])

In [23]:
# going to encode the categorical variables for the test set 
test[cat_variables] = encoder.fit_transform(test[cat_variables])

In [24]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,YrSold,SaleType,SaleCondition,SalePrice,Total_SF,Total_Bathrooms,Outdoor_Space,House_Age,Years_Since_Remodel,Nbhd_Price_Point
0,1,60,3.0,65.0,8450,1.0,NaN,3.0,3.0,0.0,...,2008,8.0,4.0,208500,2566,3.5,61,5,5,197200.0
1,2,20,3.0,80.0,9600,1.0,NaN,3.0,3.0,0.0,...,2007,8.0,4.0,181500,2524,2.0,298,31,31,218000.0
2,3,60,3.0,68.0,11250,1.0,NaN,0.0,3.0,0.0,...,2008,8.0,4.0,223500,2706,3.5,42,7,6,197200.0
3,4,70,3.0,60.0,9550,1.0,NaN,0.0,3.0,0.0,...,2006,8.0,0.0,140000,2473,2.0,35,91,36,200624.0
4,5,60,3.0,84.0,14260,1.0,NaN,0.0,3.0,0.0,...,2008,8.0,4.0,250000,3343,3.5,276,8,8,301500.0


In [25]:
# there are still alot of Nan values left so i need to clean those up
pd.set_option('display.max_rows', None)
na_count = train.isnull().sum().sort_values(ascending=False)
print(na_count)

Alley                  1369
Fence                  1179
MasVnrType              872
FireplaceQu             690
LotFrontage             259
GarageYrBlt              81
GarageCond               81
GarageQual               81
GarageType               81
GarageFinish             81
BsmtFinType2             38
BsmtExposure             38
BsmtFinType1             37
BsmtCond                 37
MasVnrArea                8
Electrical                1
Fireplaces                0
Id                        0
TotRmsAbvGrd              0
KitchenQual               0
KitchenAbvGr              0
BedroomAbvGr              0
HalfBath                  0
FullBath                  0
Functional                0
GarageArea                0
GarageCars                0
SaleCondition             0
Years_Since_Remodel       0
House_Age                 0
Outdoor_Space             0
Total_Bathrooms           0
Total_SF                  0
SalePrice                 0
SaleType                  0
GrLivArea           

In [26]:
# for all the missing variables going to impute the mean for the category
train = train.fillna(train.mean(numeric_only=True))
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,YrSold,SaleType,SaleCondition,SalePrice,Total_SF,Total_Bathrooms,Outdoor_Space,House_Age,Years_Since_Remodel,Nbhd_Price_Point
0,1,60,3.0,65.0,8450,1.0,0.450549,3.0,3.0,0.0,...,2008,8.0,4.0,208500,2566,3.5,61,5,5,197200.0
1,2,20,3.0,80.0,9600,1.0,0.450549,3.0,3.0,0.0,...,2007,8.0,4.0,181500,2524,2.0,298,31,31,218000.0
2,3,60,3.0,68.0,11250,1.0,0.450549,0.0,3.0,0.0,...,2008,8.0,4.0,223500,2706,3.5,42,7,6,197200.0
3,4,70,3.0,60.0,9550,1.0,0.450549,0.0,3.0,0.0,...,2006,8.0,0.0,140000,2473,2.0,35,91,36,200624.0
4,5,60,3.0,84.0,14260,1.0,0.450549,0.0,3.0,0.0,...,2008,8.0,4.0,250000,3343,3.5,276,8,8,301500.0


In [27]:
# now also going to impute the mean for missings in the test dataset 
test = test.fillna(test.mean(numeric_only=True))

In [28]:
# lets make sure the train and test set are the same 
print(train.shape)
print(test.shape)

(1460, 78)
(1459, 77)


In [29]:
# we see that there is a column missing in the test set so now need to list column names to determine what column is missing 
print(train.columns)
print(test.columns)

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir',
       'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
       'BsmtFullBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces',
       'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish',
       'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'ScreenPor

In [30]:
# going to do a log transformation on the target variable 
sns.histplot(train['SalePrice'])

<Axes: xlabel='SalePrice', ylabel='Count'>

In [31]:
# train['SalePrice'] = np.log(train['SalePrice'])
# sns.histplot(train['SalePrice'])

In [32]:
train.dtypes

Id                       int64
MSSubClass               int64
MSZoning               float64
LotFrontage            float64
LotArea                  int64
Street                 float64
Alley                  float64
LotShape               float64
LandContour            float64
Utilities              float64
LotConfig              float64
LandSlope              float64
Neighborhood           float64
Condition1             float64
Condition2             float64
BldgType               float64
HouseStyle             float64
OverallQual              int64
OverallCond              int64
YearBuilt                int64
YearRemodAdd             int64
RoofStyle              float64
RoofMatl               float64
Exterior1st            float64
Exterior2nd            float64
MasVnrType             float64
MasVnrArea             float64
ExterQual              float64
ExterCond              float64
Foundation             float64
BsmtCond               float64
BsmtExposure           float64
BsmtFinT

***First Model using Random Forest***

In [33]:
# aniyahs model - going to do a random forest
# since we dont have a y_test(test['sale price']) going to split the training into a train and validate set (validate will be used to actually look at performance since we dont have the actual sale price)
X = train.drop('SalePrice',axis=1)
y= train['SalePrice']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size = 0.20, random_state=42
)
# now that ive split the data going to train my model 
RF = RandomForestRegressor(n_estimators= 200, max_depth=None, random_state=0)
RF.fit(X_train, y_train)
#predicting on the validation set
RF_preds = RF.predict(X_val)
# calculating RMSE for the validation set
RF_rmse = np.sqrt(mean_squared_error(y_val, RF_preds))
print(np.log(RF_rmse)) # i believe RMSE is calculated on a log scale

10.29850452420011


***Second Model***

Using an initial base model to build upon via recursive feature elimination

In [44]:
# yani's base model   

xg_model = GradientBoostingRegressor(
    n_estimators=100,    
    learning_rate=0.15,  
    max_depth=10,         
    min_samples_leaf=1,
    random_state=42
)

xg_model.fit(X_train, y_train)

XG_preds = xg_model.predict(X_val)
XG_rmse=np.sqrt(mean_squared_error(y_val, XG_preds))
print(np.log(XG_rmse))

10.47374276395517


Note, very few features have strong correlations with Sales Price. This may be a sign to narrow down the features used.

In [45]:
# Check top correlations
print(train.corr()['SalePrice'].sort_values(ascending=False).head(10))

SalePrice           1.000000
OverallQual         0.790982
Total_SF            0.782260
Nbhd_Price_Point    0.733515
GrLivArea           0.708624
GarageCars          0.640409
Total_Bathrooms     0.628219
GarageArea          0.623431
TotalBsmtSF         0.613581
1stFlrSF            0.605852
Name: SalePrice, dtype: float64


In [46]:
from sklearn.feature_selection import RFE


selector = RFE(xg_model, n_features_to_select=15, step=1)
selector = selector.fit(X_train, y_train)

# See which ones survived
rankings = pd.DataFrame({'Feature': X_train.columns, 'Rank': selector.ranking_})
top_features = rankings[rankings['Rank'] == 1]['Feature'].values
print(top_features)

['LotFrontage' 'LotArea' 'OverallQual' 'YearBuilt' 'BsmtFinSF1'
 'BsmtUnfSF' '1stFlrSF' '2ndFlrSF' 'GrLivArea' 'GarageYrBlt' 'GarageArea'
 'MoSold' 'Total_SF' 'Years_Since_Remodel' 'Nbhd_Price_Point']


Building subsequent xgboost model using the top 15 features

In [63]:
cols=['LotFrontage', 'LotArea', 'OverallQual', 'YearBuilt', 'BsmtFinSF1',
 'BsmtUnfSF', 'GrLivArea', 'GarageYrBlt', 'GarageArea',
 'MoSold', 'Total_SF', 'Nbhd_Price_Point']

ID_train= X_train['Id']
ID_val= X_val['Id']

X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X, y, test_size = 0.20, random_state=42
)
X_train_new=X_train_new[cols]
X_val_new=X_val_new[cols]

y_train_log = np.log1p(y_train_new)
y_val_log = np.log1p(y_val_new)

xg_model_new = GradientBoostingRegressor(
    n_estimators=500,    
    learning_rate=0.15,  
    max_depth=4,         
    min_samples_leaf=1,
    random_state=42
)

xg_model_new.fit(X_train_new, y_train_log)

log_preds = xg_model_new.predict(X_val_new)
                       
log_rmse = np.sqrt(mean_squared_error(y_val_log, log_preds))
print((log_rmse))

0.15739593506183863


In [35]:
# randa's model

## Reference for how to stack the models 
1. https://www.geeksforgeeks.org/machine-learning/stacking-in-machine-learning/

In [36]:
# importing the package to stack
from sklearn.ensemble import StackingRegressor

In [37]:
# stacking the models together 
estimators = [ model, 
              # my model 
              # randa's model

]

stacked_model = StackingRegressor(estimators=estimators)